In [1]:
import pandas as pd
import datetime as dt

In [2]:
fact_sales = pd.read_csv('../data/Fact_Sales.csv')
dim_customer = pd.read_csv('../data/Dim_Customer.csv')

fact_sales['Order Date'] = pd.to_datetime(fact_sales['Order Date'])

# Hợp nhất dữ liệu (Merge) để lấy thông tin kênh bán
df_rfm = pd.merge(fact_sales, dim_customer, on='Customer ID', how='left')

Tính toán các chỉ số R, F, M
- Recency (Độ gần đây): Số ngày kể từ lần cuối khách hàng mua sữa. Khách mua càng gần đây, điểm càng cao.
- Frequency (Tần suất): Số lần đặt hàng. Khách đại lý (B2B) thường có tần suất cao hơn khách lẻ (DTC).
- Monetary (Giá trị): Tổng doanh thu mang lại.

In [3]:
# Lấy mốc thời gian là ngày mua hàng cuối cùng trong data + 1 ngày
snapshot_date = df_rfm['Order Date'].max() + dt.timedelta(days=1)

rfm = df_rfm.groupby(['Customer ID', 'Sales_Channel']).agg({
    'Order Date': lambda x: (snapshot_date - x.max()).days, # Recency
    'Order ID': 'nunique',                                  # Frequency
    'Sales': 'sum'                                          # Monetary
}).reset_index()

rfm.rename(columns={'Order Date': 'Recency', 
                    'Order ID': 'Frequency', 
                    'Sales': 'Monetary'}, inplace=True)

In [4]:
# Chấm điểm RFM (Chia quartile từ 1 đến 4)
# Recency: Số ngày càng nhỏ (càng gần) thì điểm càng cao (labels=[4,3,2,1])
# Frequency & Monetary: Càng lớn càng cao (labels=[1,2,3,4])

r_labels = range(4, 0, -1)
f_labels = range(1, 5)
m_labels = range(1, 5)

rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=r_labels)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=f_labels)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=m_labels)

# Gộp chuỗi điểm RFM (Ví dụ: 444 là VIP nhất, 111 là tệ nhất)
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

Customer Segmentation
- Champions: VIP
- Loyal Customers: Trung thành
- Needs Attention: Cần chú ý
- At Risk: Nguy cơ rời bỏ
- Hibernating: Đang ngủ đông

In [5]:
#  Phân loại Khách hàng (Customer Segmentation)

def assign_segment(df):
    if df['RFM_Score'] >= 10:
        return 'Champions'
    elif (df['RFM_Score'] >= 8) and (df['RFM_Score'] < 10):
        return 'Loyal Customers'
    elif (df['RFM_Score'] >= 6) and (df['RFM_Score'] < 8):
        return 'Needs Attention'
    elif (df['RFM_Score'] >= 4) and (df['RFM_Score'] < 6):
        return 'At Risk'
    else:
        return 'Hibernating'

rfm['Customer_Status'] = rfm.apply(assign_segment, axis=1)

In [6]:
print(rfm['Customer_Status'].value_counts())

rfm.to_csv('../data/RFM_Analysis.csv', index=False)

Customer_Status
Loyal Customers    222
Champions          193
Needs Attention    182
At Risk            141
Hibernating         55
Name: count, dtype: int64
